In [1]:
pip install PyPDF2 pandas openpyxl


  Obtaining dependency information for PyPDF2 from https://files.pythonhosted.org/packages/8e/5e/c86a5643653825d3c913719e788e41386bee415c2b87b4f955432f2de6b2/pypdf2-3.0.1-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/232.6 kB ? eta -:--:--
   ------------------------------------- - 225.3/232.6 kB 13.4 MB/s eta 0:00:01
   ---------------------------------------- 232.6/232.6 kB 7.2 MB/s eta 0:00:00



[notice] A new release of pip is available: 23.2.1 -> 24.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import PyPDF2
import pandas as pd
import re
from collections import Counter
import requests
from io import BytesIO

# Load positive and negative words from Excel
word_df = pd.read_excel(r"C:\Users\rohit\Desktop\sentiment.xlsx")
positive_words = set(word_df['Positive words'].dropna().str.lower().tolist())
negative_words = set(word_df['Negative words'].dropna().str.lower().tolist())

# Function to extract text from a PDF link
def extract_text_from_pdf(pdf_link):
    response = requests.get(pdf_link)
    pdf_file = BytesIO(response.content)
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    text = ''
    for page in pdf_reader.pages:
        text += page.extract_text()
    return text

# Function to perform sentiment analysis on the extracted text
def sentiment_analysis(text):
    # Normalize the text
    words = re.findall(r'\b\w+\b', text.lower())
    total_words = len(words)
    
    # Count positive and negative words
    word_counts = Counter(words)
    positive_count = sum(word_counts[word] for word in positive_words if word in word_counts)
    negative_count = sum(word_counts[word] for word in negative_words if word in word_counts)
    
    # Calculate sentiment index
    sentiment_index = (negative_count - positive_count) / total_words if total_words > 0 else 0
    
    return positive_count, negative_count, total_words, sentiment_index

# List of PDF links
pdf_links = ["https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/0FSRJUN2024_270620242B95CB128D1847A3ACAB5B5A4BEBF0DF.PDF",
             "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/0FSRDECB815B9437D6D428F81D45C22BBF6C62A.PDF",
             "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/0FSRJUNE20231159B36F45EA406E9D704BBC8F73D785.PDF",
             "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSRDECEMBER2022F93A2F188A394ACDB2FDDC2FCC0D07F0.PDF",
             "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/0FSRJUNE2022F758BFB27A9145A385FE9AC8D204AC82.PDF",
             "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/FSRDEC2021_FULL2D99E6548CD0478CA90EE717F2B85D45.PDF",
             "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/FSRJULY20210595CD3BEDFA466EBE9169BCE426E32C.PDF",
             "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/FSR_F06B552BF8B144B80B4AEFEDEB3D62218.PDF",
             "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSRJULY2020C084CED43CD1447D80B4789F7E49E499.PDF",
             "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSRDECEMBER20198C840246658946159CB3B94E8516F2EC.PDF",
             "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/FSRJUNE2019E5ECDDAD7E514756AFEF1E71CB2ADA2B.PDF",
             "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSRDECEMBER2018DAFEDD89C01C432786925639A4864F96.PDF",
             "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSR_JUNE2018A3526EF7DC8640539C1420D256A470FC.PDF",
             "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/0FSR201730210986ADDA44E2A946A3F6C4408581.PDF",
             "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSR_30061794092D8D036447928A4B45880863B33E.PDF",
             "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSR_166BABD6ABE04B48AFB534749A1BF38882.PDF",
             "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSR2316BB76DB39BF964542B9D1EBE2CBC273E7.PDF",
             "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSR6F7E7BC6C14F42E99568A80D9FF7BBA6.PDF",
             "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/0FS15A56030B88BD047B4A7124BA5AF1D8CF2.PDF",
             "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/FSR29122014_FL.PDF",
             "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSR26062014F.pdf",
             "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/0FSR26062014F.pdf",
             "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/FSRDEC301213_FL.pdf",
             "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/FSPI260613FL.pdf",
             "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/FFSR261212_FL.pdf",
             "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/FFSR260612_FL.pdf",
             "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/FSRD22122011_F.pdf",
             "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/FSR140611FL.pdf",
             "https://rbidocs.rbi.org.in/rdocs//PublicationReport/Pdfs/FSR301210F.pdf",
             "https://rbidocs.rbi.org.in/rdocs/PublicationReport/Pdfs/IFSR250310F.pdf",
            ]

# Data storage for CSV
data = []

# Iterate through the PDF links and perform sentiment analysis
for link in pdf_links:
    text = extract_text_from_pdf(link)
    
    # Extract date (month-year) from the first page
    match = re.search(r'(\bJanuary|\bFebruary|\bMarch|\bApril|\bMay|\bJune|\bJuly|\bAugust|\bSeptember|\bOctober|\bNovember|\bDecember)\s+\d{4}', text, re.IGNORECASE)
    date = match.group(0) if match else 'Unknown'
    
    # Perform sentiment analysis
    positive_count, negative_count, total_words, sentiment_index = sentiment_analysis(text)
    
    # Append data
    data.append([date, positive_count, negative_count, total_words, sentiment_index])

# Save the results to a CSV file
output_df = pd.DataFrame(data, columns=['Date', 'Total Positive Words', 'Total Negative Words', 'Total Words', 'Sentiment Index'])
output_df.to_csv('sentiment_analysis_results.csv', index=False)

print("Sentiment analysis completed and saved to 'sentiment_analysis_results.csv'.")


Sentiment analysis completed and saved to 'sentiment_analysis_results.csv'.
